In [ ]:
import pandas as pd
import numpy as np

In [ ]:
lookup= pd.read_csv('sa2_lookup.csv',dtype={'SA2_CODE_2021': str})
lookup

In [ ]:
lookup.head(10)

In [ ]:
lookup['GCC_NAME'].value_counts()

In [ ]:
adelaide= lookup[lookup['GCC_NAME']=='Greater Adelaide'].copy()
adelaide

In [ ]:
go2=pd.read_csv("C:\Users\kiran\Downloads\2021_GCP_SA2_for_SA_short-header\2021 Census GCP Statistical Area 2 for SA\2021Census_G02_SA_SA2.csv",
                dtype{='SA_CODE_2021': str})
go2

In [ ]:
# we need to add r before opening quote
g02=pd.read_csv(r"C:\Users\kiran\Downloads\2021_GCP_SA2_for_SA_short-header\2021 Census GCP Statistical Area 2 for SA\2021Census_G02_SA_SA2.csv",
                dtype={'SA2_CODE_2021': str})
g02

In [ ]:
# rename ugly column names
g02 = g02.rename(columns={
    'SA2_CODE_2021': 'SA2_CODE_2021',           
    'Median_age_persons': 'median_age',
    'Median_mortgage_repay_monthly': 'median_mortgage',
    'Median_tot_prsnl_inc_weekly': 'weekly_personal_income',
    'Median_rent_weekly': 'weekly_rent',
    'Median_tot_fam_inc_weekly': 'weekly_family_income',
    'Median_tot_hhd_inc_weekly': 'weekly_household_income',
    'Average_num_psns_per_bedroom': 'avg_persons_per_bedroom',
    'Average_household_size': 'avg_household_size'
})
# Keep only columns we need (drop the rest)
cols_to_keep = ['SA2_CODE_2021', 'median_age', 'weekly_personal_income',
                'weekly_household_income', 'weekly_rent', 'avg_household_size']

g02 = g02[cols_to_keep]
g02

In [ ]:
# load g01 data as it contains population data
g01=pd.read_csv(r"C:\Users\kiran\Downloads\2021_GCP_SA2_for_SA_short-header\2021 Census GCP Statistical Area 2 for SA\2021Census_G01_SA_SA2.csv",
                dtype={'SA2_CODE_2021':str})
g01

In [ ]:
# we have more columns but we need only total population columns
pop=g01[['SA2_CODE_2021', 'Tot_P_P']].copy()
pop=pop.rename(columns={'Tot_P_P' : 'population'})
pop

In [ ]:
# now we need to load store data to know which suburbs does store already exists
stores=pd.read_csv(r"C:\Users\kiran\Downloads\bunnings_sa_stores.csv")
stores = stores[stores['suburb'] != 'Gepps Cross']
stores

In [ ]:
# now its time to merge all files into one master file
master = adelaide[['SA2_CODE_2021', 'SA2_NAME_2021', 'SA3_NAME']].copy()
# we merge g02 to get income
master = master.merge(g02, on='SA2_CODE_2021', how='left')
# merge pop to get population
master = master.merge(pop, on='SA2_CODE_2021', how='left')
master

In [ ]:
# now we need to merge bunnings count but we focusing adelaide metro stores
stores_adelaide = stores[stores['is_adelaide_metro'] == 'Yes'].copy()
#mapping store suburb names to ABS SA2 name
name_mapping = {
    'Adelaide Airport':  'Adelaide Airport',
    'Edwardstown':       'Edwardstown',
    'Gawler':            'Gawler - South',
    'Kent Town':         'Norwood (SA)',
    'Oaklands Park':     'Warradale',
    'Mile End':          'Richmond (SA)',
    'Modbury':           'Hope Valley - Modbury',
    'Mount Barker':      'Mount Barker',
    'Munno Para West':   'Munno Para West - Angle Vale',
    'Noarlunga Centre':  'Christies Beach',
    'Parafield':         'Parafield',
    'Prospect':          'Prospect',
    'Reynella':          'Reynella',
    'Seaford Meadows':   'Seaford - Seaford Meadows',
    'Windsor Gardens':   'Windsor Gardens',
    'Woodville':         'Woodville - Cheltenham',
}

stores_adelaide['sa2_matched'] = stores_adelaide['suburb'].map(name_mapping)
stores_adelaide

In [ ]:
# Count per SA2
store_counts = (
    stores_adelaide
    .groupby('sa2_matched') # put stores into buckets- one bucket for one suburb
    .size()                  # cpunt how many stores in each bucket
    .reset_index              # turn result into proper table(not a series)
    (name='bunnings_count') # name coulumn ' bunnings_count'
    .rename(columns={'sa2_matched': 'SA2_NAME_2021'})) 

# Merge into master
master = master.merge(store_counts, on='SA2_NAME_2021', how='left')
master['bunnings_count'] = master['bunnings_count'].fillna(0).astype(int)
master

In [ ]:
print(f"Suburbs WITH Bunnings: {(master['bunnings_count'] > 0).sum()}")
print(f"Suburbs WITHOUT Bunnings: {(master['bunnings_count'] == 0).sum()}")

In [ ]:
!pip install pyodbc sqlalchemy 

In [ ]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine
from urllib.parse import quote_plus

server = "LAPTOP-QJVGB01O\SQLEXPRESS"
database= "adelaide_retail"

# Build the connection string for Windows Authentication
conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

# Wrap it so SQLAlchemy + pandas can use it
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}")


In [ ]:
# Test it!
test = pd.read_sql("SELECT @@VERSION AS sql_server_version", engine)
print(" Connected to SQL Server!")
print(test.iloc[0, 0])

In [ ]:
#pushing master dataframe into ssms
master.to_sql('suburbs', engine, if_exists='replace', index=False)
check = pd.read_sql("SELECT COUNT(*) AS row_count FROM suburbs", engine)
print(f"Rows now in SQL Server: {check['row_count'][0]}")